# BSF Larvae Detection - GPU Version
This notebook processes Black Soldier Fly larvae images using YOLO detection with GPU support.

**Key Features:**
- GPU-accelerated YOLO inference
- EasyOCR for tray ID detection
- Automated data upload to API
- Designed for cloud GPU platforms (Vast.ai, RunPod, Colab)

## 1. Setup Environment
Install required packages and configure GPU settings

In [ ]:
# Install required packages (run once)
!pip install ultralytics easyocr opencv-python pillow numpy requests

# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Import Libraries
Import all required modules for image processing, YOLO detection, OCR, and API communication

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
from PIL import Image
import requests
import json
import os
from pathlib import Path
from datetime import datetime

print("✅ All libraries imported successfully!")

## 3. Configuration
Set up paths, API endpoints, and model parameters

In [ ]:
# API Configuration
API_URL = "http://134.122.83.13:8000/api/upload"  # Your server URL
USERNAME = "your_username"  # Replace with your username
PASSWORD = "your_password"  # Replace with your password

# Model Configuration
MODEL_PATH = "best.pt"  # Path to your YOLO model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use GPU if available

# Image Processing Configuration
IMG_FOLDER = "./img"  # Folder containing images to process
OUTPUT_FOLDER = "./output"  # Folder to save processed images

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"✅ Configuration complete!")
print(f"   Device: {DEVICE}")
print(f"   Image folder: {IMG_FOLDER}")
print(f"   API URL: {API_URL}")

## 4. Load YOLO Model
Initialize the YOLO model for larvae detection on GPU

In [ ]:
# Load YOLO model
print("Loading YOLO model...")
model = YOLO(MODEL_PATH)
model.to(DEVICE)  # Move model to GPU
print(f"✅ YOLO model loaded on {DEVICE}")

# Initialize EasyOCR reader for tray ID detection
print("Loading EasyOCR...")
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
print("✅ EasyOCR loaded")

## 5. Helper Functions
Define functions for tray ID detection and data upload

In [ ]:
def detect_tray_id(image_path):
    """Detect tray ID using OCR from the image"""
    try:
        image = cv2.imread(str(image_path))
        results = reader.readtext(image)
        
        for (bbox, text, prob) in results:
            text = text.strip().replace(" ", "").replace("O", "0")
            if text.isdigit() and len(text) >= 4:
                return int(text)
        
        print(f"⚠️  No tray ID detected in {image_path}")
        return None
    except Exception as e:
        print(f"❌ OCR error: {e}")
        return None


def upload_to_api(larvae_data, tray_number):
    """Upload larvae detection data to the server API"""
    try:
        payload = {
            "username": USERNAME,
            "password": PASSWORD,
            "tray_number": tray_number,
            "larvae_data": larvae_data
        }
        
        response = requests.post(API_URL, json=payload, timeout=30)
        
        if response.status_code == 200:
            print(f"✅ Data uploaded successfully for Tray {tray_number}")
            return True
        else:
            print(f"❌ Upload failed: {response.status_code} - {response.text}")
            return False
    except Exception as e:
        print(f"❌ Upload error: {e}")
        return False


print("✅ Helper functions defined")

## 6. Process Images
Main processing loop - detect larvae and upload data

In [ ]:
# Get list of images to process
image_files = list(Path(IMG_FOLDER).glob("*.jpg")) + list(Path(IMG_FOLDER).glob("*.png"))
print(f"Found {len(image_files)} images to process")

# Process each image
for idx, img_path in enumerate(image_files, 1):
    print(f"\n{'='*60}")
    print(f"Processing image {idx}/{len(image_files)}: {img_path.name}")
    print(f"{'='*60}")
    
    # Detect tray ID
    tray_id = detect_tray_id(img_path)
    if not tray_id:
        print(f"⏭️  Skipping image (no tray ID detected)")
        continue
    
    print(f"📋 Tray ID: {tray_id}")
    
    # Run YOLO detection (GPU accelerated!)
    print("🔍 Running YOLO detection...")
    results = model.predict(source=str(img_path), device=DEVICE, verbose=False)
    
    # Extract larvae data
    larvae_data = []
    for result in results:
        boxes = result.boxes
        
        for box in boxes:
            # Get bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            
            # Calculate measurements
            length = float(y2 - y1)
            width = float(x2 - x1)
            area = length * width
            
            # Estimate weight (based on your calibration)
            weight = area * 0.000001  # Adjust this multiplier based on your data
            
            larvae_data.append({
                "length": round(length, 2),
                "width": round(width, 2),
                "area": round(area, 2),
                "weight": round(weight, 6)
            })
    
    print(f"✅ Detected {len(larvae_data)} larvae")
    
    # Upload to API
    if larvae_data:
        upload_to_api(larvae_data, tray_id)
    else:
        print("⚠️  No larvae detected, skipping upload")

print(f"\n{'='*60}")
print(f"✅ Processing complete! Processed {len(image_files)} images")
print(f"{'='*60}")